# Spatial-spectral mixing with NewAthena X-IFU

This notebook demonstrates why adjacent extended-source regions cannot always be fitted independently. It preserves the 2026 SIXTE workshop east/soft and west/hard experiment, uses the NewAthena X-IFU baseline response, and compares naive regional fits with a simultaneous four-response fit.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from newathena_sixte_extended_sources import load_workspace

WORKSPACE = load_workspace()
ROOT = WORKSPACE.root
EXPOSURE = int(WORKSPACE.profile_values['phase3_exposure_s'])
ARF_PHOTONS = int(WORKSPACE.profile_values['phase3_arf_photons'])
RUNTIME = WORKSPACE.runtime / 'phase3-mixing'
required = [
    RUNTIME / 'mixing_soft.simput', RUNTIME / 'mixing_hard.simput',
    RUNTIME / f'mixing_{EXPOSURE}s_east.pha',
    RUNTIME / f'mixing_{EXPOSURE}s_west.pha',
    RUNTIME / 'phase3_region_source_mixing.csv',
    RUNTIME / f'phase3_{EXPOSURE}s_naive_fit_results.json',
    RUNTIME / f'phase3_{EXPOSURE}s_n{ARF_PHOTONS}_mixed_fit_results.json',
]
for path in required:
    WORKSPACE.require(path, f'Phase 3 {WORKSPACE.profile} product')
print({'profile': WORKSPACE.profile, 'exposure_s': EXPOSURE,
       'arf_photons': ARF_PHOTONS,
       'science_status': WORKSPACE.profile_values['science_status']})

## Experiment contract

The source is a flat three-arcminute circle split into two hemispheres. The east source has an absorbed power law with photon index 2.5; the west source has index 1.5. Each has the same 2--10 keV energy flux, $2\times10^{-11}$ erg s$^{-1}$ cm$^{-2}$, and $N_H=10^{22}$ cm$^{-2}$. The simulation uses 100 ks, no instrumental background, and seed 20260722.

In [ ]:
for component in ('soft', 'hard'):
    with fits.open(RUNTIME / f'mixing_{component}.simput') as hdus:
        source = hdus['SRC_CAT'].data[0]
        spectrum = hdus['SPECTRUM'].data[0]
        print(component, {'source_id': int(source['SRC_ID']), 'flux': float(source['FLUX']), 'spectral_bins': len(spectrum['ENERGY'])})

## Detected source-to-region mixing

Source IDs are retained in the simulated event list, so the contamination matrix is measured directly rather than inferred from fitted spectra.

In [ ]:
mixing = pd.read_csv(RUNTIME / 'phase3_region_source_mixing.csv')
mixing_selected = mixing.query('exposure_s == @EXPOSURE').pivot(
    index='region', columns='source', values='detected_fraction')
display(mixing_selected.style.format('{:.3%}'))
summary = json.loads((RUNTIME / 'phase3_mixing_summary.json').read_text())
assert all(item['pha_count_reconciliation'] for item in summary['exposures'][str(EXPOSURE)]['regions'].values())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
image = ax.imshow(mixing_selected.to_numpy(), vmin=0, vmax=1, cmap='Blues')
ax.set_xticks(range(2), mixing_selected.columns)
ax.set_yticks(range(2), mixing_selected.index)
ax.set_xlabel('True source')
ax.set_ylabel('Extraction region')
for row in range(2):
    for column in range(2):
        value = mixing_selected.iloc[row, column]
        ax.text(column, row, f'{value:.2%}', ha='center', va='center', color='white' if value > 0.5 else 'black')
fig.colorbar(image, ax=ax, label='Detected fraction')
fig.tight_layout()

## ARF convergence

Four ARFs map each true source into each extraction region. The production trial uses 100,000 photons per sampled ARF bin. Dominant paths are sub-percent stable; the much weaker cross paths retain percent-level sampling uncertainty.

In [ ]:
arf_summary = json.loads((RUNTIME / 'phase3_arf_convergence.json').read_text())
convergence = pd.DataFrame(arf_summary['comparisons'])
if WORKSPACE.profile == 'reference':
    selected_convergence = convergence.query('higher_photon_trial == @ARF_PHOTONS').copy()
else:
    selected_convergence = pd.DataFrame({'note': [
        'The 10k teaching ARFs demonstrate the method; convergence is assessed with the reference 50k-to-100k comparison.']})
selected_convergence

## Naive versus mixed-response inference

The naive fit assigns one absorbed power law and the standard on-axis response to each region. The mixed fit instead applies both intrinsic source models to both spectra through the four source-to-region ARFs.

In [ ]:
naive = pd.DataFrame(json.loads(
    (RUNTIME / f'phase3_{EXPOSURE}s_naive_fit_results.json').read_text())).set_index('region')
mixed_raw = json.loads(
    (RUNTIME / f'phase3_{EXPOSURE}s_n{ARF_PHOTONS}_mixed_fit_results.json').read_text())
mixed = pd.DataFrame(mixed_raw['models']).set_index('source')
comparison = pd.DataFrame({
    'truth': [2.5, 1.5],
    'naive regional fit': [naive.loc['east', 'photon_index'], naive.loc['west', 'photon_index']],
    'mixed-response fit': [mixed.loc['soft', 'photon_index'], mixed.loc['hard', 'photon_index']],
}, index=['soft / east', 'hard / west'])
display(comparison.style.format('{:.5f}'))
if WORKSPACE.profile == 'reference':
    assert np.max(np.abs(comparison['mixed-response fit'] / comparison['truth'] - 1)) < 0.001

In [ ]:
ax = comparison.plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel('Photon index')
ax.set_title('Explicit response mixing removes regional slope bias')
ax.legend(frameon=False)
plt.tight_layout()

## Try it: why the contamination is asymmetric

**Exercise.** Both sources have equal 2--10 keV energy flux. Explain why the soft source contributes a larger fraction of west-region counts than the hard source contributes to east-region counts.

<details><summary>Solution and interpretation</summary>

A steeper power law contains more photons per unit energy flux in the X-IFU band. The geometrical leakage can therefore be similar while the detected count contamination is asymmetric. Teaching-mode ARFs are intentionally noisy and demonstrate wiring only; use the reference profile for response-convergence and sub-percent truth-recovery claims.
</details>

## Interpretation and limitations

Equal energy fluxes produce unequal detected photon counts because the soft spectrum contains more photons in the X-IFU band. Consequently, the soft leakage has a larger effect on the hard west spectrum than hard leakage has on the east spectrum. The simultaneous response-mixing fit recovers both input slopes to about 0.05% in this realization. This numerical recovery does not erase the remaining Monte Carlo uncertainty in the weak cross-region ARFs, and the simulation excludes instrumental background and more complex plasma emission.